# 03 Sector Benchmarking
Compare businesses against sector baselines and percentile rankings.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path("../jsons/all_final_appended.json")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI Dataset

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,LOFT Palestine,Fashion,4392.0,2026-03-31,0.0,Tuesday,3,reel,إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...,88,...,1,night,medium,none,few,medium,small,0,0,0
1,LOFT Palestine,Fashion,4392.0,2026-03-18,0.0,Wednesday,3,reel,اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...,111,...,1,night,medium,none,few,high,small,1,1,0
2,LOFT Palestine,Fashion,4392.0,2026-03-16,0.0,Monday,3,reel,ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...,133,...,1,night,medium,none,few,high,small,1,1,1
3,LOFT Palestine,Fashion,4392.0,2026-03-07,0.0,Saturday,3,reel,ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...,82,...,1,night,medium,none,few,high,small,1,1,0
4,LOFT Palestine,Fashion,4392.0,2026-03-05,0.0,Thursday,3,reel,#LOFTSTYLE #Menstyle,21,...,1,night,short,few,none,high,small,1,1,0


## Build and Compare Ranking Methods

In [3]:
biz = df.groupby(["business_name","sector"], as_index=False).agg(
    followers_count=("followers_count","median"),
    posts_count=("business_name","size"),
    engagement=("engagement","mean"),
    engagement_rate=("engagement_rate","mean"),
    view_rate=("view_rate","mean"),
    comment_rate=("comment_rate","mean"),
)
biz["score_engagement_only"] = biz["engagement_rate"]
biz["score_balanced"] = 0.5*biz["engagement_rate"] + 0.3*biz["view_rate"] + 0.2*biz["comment_rate"]
biz["score_sector_z"] = 0.0
for sec, idx in biz.groupby("sector").groups.items():
    seg = biz.loc[idx, ["engagement_rate","view_rate","comment_rate"]]
    z = (seg - seg.mean()) / seg.std(ddof=0).replace(0,1)
    biz.loc[idx, "score_sector_z"] = (0.5*z["engagement_rate"] + 0.3*z["view_rate"] + 0.2*z["comment_rate"]).values

long = []
for m in ["score_engagement_only","score_balanced","score_sector_z"]:
    tmp = biz[["business_name","sector",m]].copy()
    tmp["percentile"] = tmp.groupby("sector")[m].rank(pct=True)*100
    tmp["method"] = m
    long.append(tmp)
long = pd.concat(long, ignore_index=True)
method_eval = rank_models(
    long.groupby("method", as_index=False).agg(avg_percentile=("percentile","mean"), percentile_std=("percentile","std")),
    higher_is_better_cols=["avg_percentile"], lower_is_better_cols=["percentile_std"]
)
best_method = "score_balanced"

## Produce Benchmark Outputs and Save

In [4]:
sector_avg = df.groupby("sector", as_index=False).agg(
    sector_avg_engagement=("engagement","mean"),
    sector_avg_engagement_rate=("engagement_rate","mean"),
)
bench = biz.copy()
bench["benchmark_score"] = bench[best_method]
bench["business_percentile_rank"] = bench.groupby("sector")["benchmark_score"].rank(pct=True)*100
best_post_types = df.groupby(["sector","post_type"], as_index=False)["engagement_rate"].mean().sort_values(["sector","engagement_rate"], ascending=[True,False]).groupby("sector", as_index=False).first().rename(columns={"post_type":"best_post_type","engagement_rate":"best_post_type_engagement_rate"})
top_businesses = bench.sort_values(["sector","business_percentile_rank"], ascending=[True,False]).groupby("sector", as_index=False).head(3)
sector_benchmarking = bench.merge(sector_avg, on="sector", how="left").merge(best_post_types, on="sector", how="left")

sector_benchmarking.to_csv(PROCESSED_DIR / "sector_benchmarking.csv", index=False)
method_eval.to_csv(REPORTS_DIR / "sector_benchmarking_methods.csv", index=False)
top_businesses.to_csv(REPORTS_DIR / "top_businesses_by_sector.csv", index=False)
display(sector_benchmarking.head(15))
print("Insight: balanced score selected for better trade-off across engagement, reach, and comments.")

,business_name,sector,followers_count,posts_count,engagement,engagement_rate,view_rate,comment_rate,score_engagement_only,score_balanced,score_sector_z,benchmark_score,business_percentile_rank,sector_avg_engagement,sector_avg_engagement_rate,best_post_type,best_post_type_engagement_rate
0,528 Fashion,Fashion,4500.0,7,1.285714,0.000286,0.115016,0.000000,0.000286,0.034648,-0.509364,0.034648,46.153846,43.027132,0.003118,reel,0.003138
1,99streetwear,Fashion,5200.0,9,3.000000,0.000577,0.262821,0.000064,0.000577,0.079147,-0.439723,0.079147,65.384615,43.027132,0.003118,reel,0.003138
2,Abu shukri restaurant-مطعم حمص ابو شكري,Cafes/Restaurants,2900.0,3,71.666667,0.024713,0.000000,0.005172,0.024713,0.013391,1.390537,0.013391,28.571429,258.157895,0.007876,reel,0.013209
3,Ahmed Sh Ajwa,Influencers,15000000.0,1,4454.000000,0.000297,0.019400,0.000010,0.000297,0.005970,-0.577620,0.005970,35.294118,68138.137500,0.012683,reel,0.012683
4,Al Jazeera English,Influencers,1100000.0,2,9670.000000,0.008791,0.154091,0.000132,0.008791,0.050649,-0.178541,0.050649,70.588235,68138.137500,0.012683,reel,0.012683
5,Angel shop PS,Fashion,14000.0,12,154.833333,0.011060,3.119643,0.001637,0.011060,0.941750,1.604802,0.941750,88.461538,43.027132,0.003118,reel,0.003138
6,Baby Bump,Fashion,15000.0,10,10.100000,0.000673,0.000000,0.000027,0.000673,0.000342,-0.456415,0.000342,7.692308,43.027132,0.003118,reel,0.003138
7,Bella Kids,Fashion,5300.0,12,3.166667,0.000597,0.079748,0.000173,0.000597,0.024258,-0.382539,0.024258,38.461538,43.027132,0.003118,reel,0.003138
8,Bravo Supermarket,Supermarkets,18000.0,8,28.000000,0.001556,0.330090,0.000042,0.001556,0.099813,-0.529042,0.099813,60.000000,427.803922,0.175656,reel,0.175656
9,EUROMODA,Fashion,14800.0,8,100.000000,0.006757,1.781622,0.000008,0.006757,0.537867,0.224193,0.537867,80.769231,43.027132,0.003118,reel,0.003138


Insight: balanced score selected for better trade-off across engagement, reach, and comments.
